## 1. CARGA DE PARÁMETROS

In [0]:
import traceback

try:
    # Definición de widgets con valores por defecto
    dbutils.widgets.text("catalog", "iowa_sales", "Catalog Name")
    dbutils.widgets.text("silver_schema", "sales_silver", "Silver Schema Name")
    dbutils.widgets.text("gold_schema", "sales_gold", "Gold Schema Name")
    dbutils.widgets.text("silver_table", "sales_cleaned", "Silver Table Name")
    dbutils.widgets.text("fact_table", "fact_sales", "Fact Table Name") 
    dbutils.widgets.text("dim_date", "dim_date", "Dim Date Name")
    dbutils.widgets.text("dim_vendor", "dim_vendor", "Dim Vendor Name")
    dbutils.widgets.text("dim_store", "dim_store", "Dim Store Name")
    dbutils.widgets.text("dim_product", "dim_product", "Dim Product Name")

    env = {
        "catalog": dbutils.widgets.get("catalog"),
        "sch_silver": dbutils.widgets.get("silver_schema"),
        "sch_gold": dbutils.widgets.get("gold_schema"),
        "tb_silver": dbutils.widgets.get("silver_table"),
        "tb_fact": dbutils.widgets.get("fact_table"),
        "dim_date": dbutils.widgets.get("dim_date"),
        "dim_vendor": dbutils.widgets.get("dim_vendor"),
        "dim_store": dbutils.widgets.get("dim_store"),
        "dim_product": dbutils.widgets.get("dim_product")
    }

    print(f"Configuración cargada. Tabla Fact destino: {env['catalog']}.{env['sch_gold']}.{env['tb_fact']}")

except Exception as e:
    print(f"Error cargando parámetros: {e}")
    raise e

## 2. CREACIÓN DE LA TABLA DE HECHOS (DDL)

In [0]:
try:
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {env['catalog']}.{env['sch_gold']}.{env['tb_fact']} (
      sale_key BIGINT GENERATED ALWAYS AS IDENTITY COMMENT 'Surrogate key for Fact',
      sale_invoice_line_no STRING COMMENT 'Degenerate Dimension - Business Key',
      
      -- Foreign Keys
      date_key INT,
      product_key BIGINT,
      store_key BIGINT,
      vendor_key BIGINT,
      
      -- Measures
      state_bottle_cost DECIMAL(10, 2),
      state_bottle_retail DECIMAL(10, 2),
      bottles_sold INT,
      sale_dollars DECIMAL(10, 2),
      volume_sold_liters DECIMAL(10, 3),
      volume_sold_gallons DECIMAL(10, 3),
      
      saved_date TIMESTAMP
    )
    USING DELTA
    COMMENT 'Fact table for Iowa Liquor Sales transactions'
    """)
    print("Estructura de la tabla Fact validada.")

except Exception as e:
    print(f"Error creando tabla Fact: {e}")
    raise e

## 3. CARGA DE LA TABLA DE HECHOS (Cruce con Dimensiones)

In [0]:
try:
    print("Iniciando cruce de capa Silver con Dimensiones Gold...")
    
    spark.sql(f"""
    MERGE INTO {env['catalog']}.{env['sch_gold']}.{env['tb_fact']} AS target
    USING (
      SELECT
        s.invoice_line_no AS sale_invoice_line_no,
        d.date_key,
        p.product_key,
        st.store_key,
        v.vendor_key,
        s.state_bottle_cost,
        s.state_bottle_retail,
        s.bottles_sold,
        s.sale_dollars,
        s.volume_sold_liters,
        s.volume_sold_gallons,
        s.saved_date
      FROM {env['catalog']}.{env['sch_silver']}.{env['tb_silver']} AS s
      
      -- Cruce Dim_Date
      LEFT JOIN {env['catalog']}.{env['sch_gold']}.{env['dim_date']} AS d
        ON s.date = d.full_date
        
      -- Cruce Dim_Product (SCD1)
      LEFT JOIN {env['catalog']}.{env['sch_gold']}.{env['dim_product']} AS p
        ON s.item_number = p.product_business_key
        
      -- Cruce Dim_Vendor (SCD1)
      LEFT JOIN {env['catalog']}.{env['sch_gold']}.{env['dim_vendor']} AS v
        ON s.vendor_number = v.vendor_business_key
        
      -- Cruce Dim_Store (SCD2)
      LEFT JOIN {env['catalog']}.{env['sch_gold']}.{env['dim_store']} AS st
        ON s.store_number = st.store_business_key 
        AND (
            -- Mapeo estricto para nuevas ventas:
            (CAST(s.date AS TIMESTAMP) >= st.valid_from AND CAST(s.date AS TIMESTAMP) < st.valid_to)
            OR 
            -- Respaldo para la carga inicial masiva histórica:
            (st.is_current = true) 
        )
    ) AS source
    ON target.sale_invoice_line_no = source.sale_invoice_line_no

    WHEN MATCHED THEN
      UPDATE SET
        target.date_key = COALESCE(source.date_key, -1),
        target.product_key = COALESCE(source.product_key, -1),
        target.store_key = COALESCE(source.store_key, -1),
        target.vendor_key = COALESCE(source.vendor_key, -1),
        target.state_bottle_cost = source.state_bottle_cost,
        target.state_bottle_retail = source.state_bottle_retail,
        target.bottles_sold = source.bottles_sold,
        target.sale_dollars = source.sale_dollars,
        target.volume_sold_liters = source.volume_sold_liters,
        target.volume_sold_gallons = source.volume_sold_gallons,
        target.saved_date = source.saved_date

    WHEN NOT MATCHED THEN
      INSERT (
        sale_invoice_line_no, date_key, product_key, store_key, vendor_key,
        state_bottle_cost, state_bottle_retail, bottles_sold, sale_dollars,
        volume_sold_liters, volume_sold_gallons, saved_date
      )
      VALUES (
        source.sale_invoice_line_no,
        COALESCE(source.date_key, -1),
        COALESCE(source.product_key, -1),
        COALESCE(source.store_key, -1),
        COALESCE(source.vendor_key, -1),
        source.state_bottle_cost, source.state_bottle_retail, source.bottles_sold,
        source.sale_dollars, source.volume_sold_liters, source.volume_sold_gallons,
        source.saved_date
      )
    """)
    print("¡Carga de Fact_Sales completada con éxito!")

except Exception as e:
    print(f"Error procesando la tabla de hechos: {e}")
    raise e

dbutils.notebook.exit("Gold fact layer processing complete.")